# 01_03_Data extraction from SNCB report #

In [10]:
from pathlib import Path

import camelot
import pandas as pd

# Set PDF file path
pdf_path = Path("../data/raw/voyageurs-montes-2024-def-1.pdf")

# Set destination directory path
DATA_DIR = Path("../data/intermediate/")

The passenger counts from the National Railway Company of Belgium (SNCB/NMBS) are handled differently from the Infrabel and Statbel open datasets. Required to calculate a new passenger-weighted punctuality metric and to enrich the punctuality fact table, <br><br>
this file is manually sourced and stored directly in the Git repository for the following reasons : 
* **Lack of a persistent Open Data Portal**: The file is hosted on a commercial website where URLs are subject to change, which would break an automated download pipeline over time.<br>
* **Data availability and continuity**: The source only provides the most recent annual count (currently, 2024). By committing the PDF to this repo, we ensure the project remains **reproducible** even if the original link disappears next year.<br>
* **Security barriers**: The SNCB website implements advanced anti-bot measures (dynamic sessions and JavaScript-based tokens) that prevent standard Python scripts from accessing the file reliably without heavy and **fragile workarounds**.

## 1. Extraction from PDF ##

In [11]:
tables = camelot.read_pdf(pdf_path, pages="1-10")

## 2. DataFrame Transformation ##

In [12]:
dfs_sncb = [table.df for table in tables]
df = pd.concat(dfs_sncb, ignore_index=True)

headers = df.iloc[0]
df.columns = headers

df = df.drop(df.index[0]).reset_index(drop=True)

df.head(100)

,Station \nGare,Gem. instappende tijdens week \nMoy. montés en semaine,Gem. instappende op zaterdag\nMoy. montés le samedi,Gem. instappende op zondag\nMoy. montés le dimanche
0,AALST,5.883,2.220,1.872
1,AALST-KERREBROEK,15,-,-
2,AALTER,2.348,622,620
3,AARSCHOT,5.804,2.243,1.758
4,AARSELE,26,-,-
...,...,...,...,...
95,BRUGGE-SINT-PIETERS,106,54,52
96,BUDA,139,-,-
97,BUGGENHOUT,581,259,322
98,BUIZINGEN,740,304,192


## 3. Export to parquet ##

In [13]:
# Export
df.to_parquet(DATA_DIR / "passengers_sncb_2024.parquet")

In [14]:
# Verification
df_to_verify = pd.read_parquet(DATA_DIR / "passengers_sncb_2024.parquet")
df.head()

,Station \nGare,Gem. instappende tijdens week \nMoy. montés en semaine,Gem. instappende op zaterdag\nMoy. montés le samedi,Gem. instappende op zondag\nMoy. montés le dimanche
0,AALST,5.883,2.220,1.872
1,AALST-KERREBROEK,15,-,-
2,AALTER,2.348,622,620
3,AARSCHOT,5.804,2.243,1.758
4,AARSELE,26,-,-
